In [1]:
import importlib
import gymnasium as gym

import env.afm_env
importlib.reload(env.afm_env)
from env.afm_env import AfmEnvironment

# gym.register(
#     id="AfmEnvironment",
#     entry_point=AfmEnvironment,
# )
#
# env = gym.make(
#     "AfmEnvironment",
#     data_file_path="environments/pt_111_small_5row_missing.npz"
# )

 PACKAGE_PATH =  /home/henry/miniforge3/envs/IProject/lib/python3.11/site-packages/ppafm
 CPP_PATH     =  /home/henry/miniforge3/envs/IProject/lib/python3.11/site-packages/ppafm/cpp


In [2]:
import tqdm
from stable_baselines3.common.callbacks import CheckpointCallback, BaseCallback
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecNormalize
import os
import datetime
from torch.nn import ReLU

TRAIN_BASE_DIR = "train_results"
train_date = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
TRAIN_DIR = os.path.join(TRAIN_BASE_DIR, train_date)


def make_env_gpu(rank=0):
    def _init():
        env = AfmEnvironment(
            surface_path="materials/pt_111_small_5row_missing.xyz",
            params_path="materials/params_code.ini",
            num_historic_data=30,
            num_actions=1,
        )
        env = Monitor(env)  #, filename=os.path.join("./logs", f"env_{rank}"))
        return env

    return _init


def make_env_load(rank=0):
    def _init():
        env = AfmEnvironment(
            data_file_path="environments/pt_111_small_5row_missing.npz",
            num_historic_data=30,
            num_actions=1,
        )
        env = Monitor(env)  #, filename=os.path.join("./logs", f"env_{rank}"))
        return env

    return _init

n_envs = 4
env_arr = [make_env_load(i) for i in range(n_envs)]
vec_env = DummyVecEnv(env_arr)
# vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.)

policy_kwargs = dict(
    net_arch=[256, 256],
    activation_fn=ReLU,
)

model = SAC(
    "MultiInputPolicy",
    vec_env,
    verbose=1,
    tensorboard_log="./tensorboard_logs_testing",
    policy_kwargs=policy_kwargs,
    gradient_steps=n_envs,
)

checkpoint_callback = CheckpointCallback(
    save_freq=10000,
    save_path=os.path.join(TRAIN_DIR, "models"),
    name_prefix="sac_afm_model_" + train_date,
    save_replay_buffer=False,
)

model.learn(
    total_timesteps=10000000,
    log_interval=1,
    progress_bar=True,
    tb_log_name=f"{n_envs}_envs",#"sac_afm_env" + train_date,
    callback=[checkpoint_callback],
)
model.save(os.path.join(TRAIN_DIR, "final_model"))
# vec_env.save(os.path.join(TRAIN_DIR, "vec_normalize.pkl"))

Using cuda device
Logging to ./tensorboard_logs_testing/4_envs_9


Output()

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 63       |
|    ep_rew_mean     | 369      |
| time/              |          |
|    episodes        | 1        |
|    fps             | 156      |
|    time_elapsed    | 1        |
|    total_timesteps | 252      |
| train/             |          |
|    actor_loss      | -7.62    |
|    critic_loss     | 1.81     |
|    ent_coef        | 0.957    |
|    ent_coef_loss   | -0.0711  |
|    learning_rate   | 0.0003   |
|    n_updates       | 148      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 64.5     |
|    ep_rew_mean     | 289      |
| time/              |          |
|    episodes        | 2        |
|    fps             | 154      |
|    time_elapsed    | 1        |
|    total_timesteps | 264      |
| train/             |          |
|    actor_loss      | -6.93    |
|    critic_loss     | 39.6     |
|    ent_coef 

KeyboardInterrupt: 

In [ ]:
from matplotlib import pyplot as plt

env = AfmEnvironment(
    data_file_path="environments/pt_111_small_rows_missing.npz",
    num_historic_data=30,
    num_actions=1,
)

# Manually test environment. Reset, step to where we'll get a proper image and scan the same height
env.reset(123)
env.step([0])
env.step([0])
env.step([0])
env.step([0])
env.step([0])

for i in range(201*201):
    env.step([100])

plt.imshow(env.generated_image.T)
plt.colorbar()
plt.show()

